In [42]:
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
import pandas as pd

In [43]:
df = pd.read_csv("heart.csv")
df.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [44]:
X = df.drop(columns=["HeartDisease"])
y = df["HeartDisease"]
cat_cols = ["Sex", "ChestPainType", "RestingECG", "ExerciseAngina", "ST_Slope"]
num_cols = [n for n in X.columns if n not in cat_cols]

In [45]:
pre = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", Pipeline([
        ("scale", StandardScaler()),
        ("pca", PCA(0.95)),
    ]), num_cols),
])

In [46]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [47]:
model_params = {
    "svc": {
        "model": SVC(gamma="auto"),
        "params": {"C": [1, 10, 20], "kernel": ["rbf", "linear"]},
    },
    "random_forest": {
        "model": RandomForestClassifier(),
        "params": {"n_estimators": [10, 20, 30, 40, 50]},
    },
    "logistic": {
        "model": LogisticRegression(max_iter=1000),
        "params": {"C": [1, 10, 20, 30]},
    },
    "gaussian_nb": {
        "model": GaussianNB(),
        "params": {},
    },
    "tree": {
        "model": DecisionTreeClassifier(),
        "params": {"criterion": ["gini", "entropy"]},
    },
}

In [51]:
scores = []

for model_name, mp in model_params.items():
    clf = GridSearchCV(mp["model"], mp["params"], cv=5)
    pipe = Pipeline([("pre", pre), ("clf", clf)])
    pipe.fit(X_train, y_train)
    scores.append({
        "model": model_name,
        "best_score": clf.best_score_,
        "best_params": clf.best_params_,
    })

In [52]:
pd.DataFrame(scores)

,model,best_score,best_params
0,svc,0.874662,"{'C': 1, 'kernel': 'rbf'}"
1,random_forest,0.861019,{'n_estimators': 50}
2,logistic,0.863759,{'C': 1}
3,gaussian_nb,0.858289,{}
4,tree,0.791576,{'criterion': 'entropy'}
